# Notebook 2: `q` Validation

This is the only notebook in the suite that uses `q` directly. It ranks the saved frozen and joint Phase C checkpoints, then selects one canonical downstream checkpoint for each branch.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/jacobposchl/model-interpretability.git"
REPO_DIR = Path("/content/model_interpretability")
IN_COLAB = "google.colab" in sys.modules

if REPO_DIR.parent.exists() and IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    from google.colab import drive
    drive.mount("/content/drive")

if REPO_DIR.exists() and str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import torch

try:
    from flow_circuits.evaluation import run_q_checkpoint_validation_experiment
    from flow_circuits.training import load_yaml_config
except ModuleNotFoundError:
    REPO_DIR = Path("/content/model_interpretability")
    if REPO_DIR.exists() and str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(Path.cwd())], check=True)
    from flow_circuits.evaluation import run_q_checkpoint_validation_experiment
    from flow_circuits.training import load_yaml_config


## Parameters


In [ ]:
CONFIG_NAME = "resnet18_z_workflow"
CONFIG_PATH = Path("configs/flow/resnet18_aligned.yaml")
Q_VALIDATION_SPLIT = "val"
TOP_K_CANDIDATES_TO_SUMMARIZE = 5
MAX_IMAGES = 1024
ANCHOR_IMAGES = 256
NEIGHBOR_TOPK = 20
FORCE_RERUN = False

DRIVE_ROOT = Path("/content/drive/MyDrive/flow_circuits") if Path("/content/drive/MyDrive").exists() else Path("artifacts/flow_circuits")
NB01_OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb01_backbone_and_z_training" / CONFIG_NAME
FROZEN_CHECKPOINT_DIR = NB01_OUTPUT_DIR / "frozen_branch"
JOINT_CHECKPOINT_DIR = NB01_OUTPUT_DIR / "joint_branch"
OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb02_q_validation" / CONFIG_NAME


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
base_config = load_yaml_config(CONFIG_PATH)
result_path = OUTPUT_DIR / "q_validation_summary.json"

if result_path.exists() and not FORCE_RERUN:
    RESULTS = json.loads(result_path.read_text(encoding="utf-8"))
else:
    RESULTS = run_q_checkpoint_validation_experiment(
        base_config=base_config,
        frozen_checkpoint_dir=FROZEN_CHECKPOINT_DIR,
        joint_checkpoint_dir=JOINT_CHECKPOINT_DIR,
        device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        validation_split=Q_VALIDATION_SPLIT,
        max_images=MAX_IMAGES,
        anchor_images=ANCHOR_IMAGES,
        neighbor_topk=NEIGHBOR_TOPK,
        top_k_candidates_to_summarize=TOP_K_CANDIDATES_TO_SUMMARIZE,
        output_path=result_path,
    )

selection_path = OUTPUT_DIR / "selected_checkpoints.json"
selection_path.write_text(json.dumps(RESULTS["selected"], indent=2), encoding="utf-8")
print(f"Saved selection artifact to: {selection_path}")


In [ ]:
print("Selected frozen checkpoint:")
print(RESULTS["selected"]["frozen"]["checkpoint_path"] if RESULTS["selected"]["frozen"] else None)
print("\nSelected joint checkpoint:")
print(RESULTS["selected"]["joint"]["checkpoint_path"] if RESULTS["selected"]["joint"] else None)
print("\nTop frozen candidates:")
for row in RESULTS["top_frozen_rows"]:
    print(f"  recall@k={row['neighbor_recall_at_k']:.4f} traj={row['trajectory_alignment_mean']:.4f} epoch={row['epoch']} -> {Path(row['checkpoint_path']).name}")
print("\nTop joint candidates:")
for row in RESULTS["top_joint_rows"]:
    print(f"  recall@k={row['neighbor_recall_at_k']:.4f} traj={row['trajectory_alignment_mean']:.4f} epoch={row['epoch']} -> {Path(row['checkpoint_path']).name}")
